In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Replace with your website pages
urls = [
    "https://www.bio-monitoring.ca",
    "https://www.bio-monitoring.ca/scientific-technical-innovation",
    "https://www.bio-monitoring.ca/about-us-1",
    "https://www.bio-monitoring.ca/mission-vision",
    "https://www.bio-monitoring.ca/contact",
]

loader = WebBaseLoader(urls)
documents = loader.load()

# Chunk the text (VERY important for RAG)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(documents)

print(f"Loaded {len(chunks)} chunks")


Loaded 18 chunks


In [17]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize the HuggingFace embedding model
# "all-mpnet-base-v2" is a good balance of quality + speed
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2",
                                  model_kwargs={"device": "mps"})

# Example: embed a query
query = "Explain how the product works"
vector = embeddings.embed_query(query)

print("Embedding vector length:", len(vector))
print("First 10 values:", vector[:10])

Embedding vector length: 768
First 10 values: [0.043266475200653076, -0.07141229510307312, -0.02030530944466591, -0.00033184344647452235, -0.09570100158452988, 0.033222496509552, 0.01032829750329256, -0.011755989864468575, -0.033142127096652985, -0.027228863909840584]


In [18]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

vectorstore.persist()

print("Vector store created and saved.")

Vector store created and saved.


/var/folders/fc/l7c3sbjd2f11ts47p5b52k240000gn/T/ipykernel_18522/1284363123.py:9: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [29]:
#Sanity check
query = "What is their innovation?"
results = vectorstore.similarity_search(query, k=18)

for i, doc in enumerate(results, 1):
    clean_string = " ".join(doc.page_content.split())
    print(f"\nResult {i}:\n{clean_string}...")



Result 1:
Scientific & Technical Innovation | bio-monitoring.ca bio-monitoring.ca Transform nature for human wellbeing bio-monitoring.ca Scientific & Technical Innovation About us Mission & Vision Contact Search 0 Wishlist 0 Cart our product»Scientific & Technical Innovation Did you know that in Canada,200 infants are born each year with neural tube defects (NTDs)? Did you know Synthetic Folic Acid Dominance & Its Limitations?...

Result 2:
Mission & Vision | bio-monitoring.ca bio-monitoring.ca Transform nature for human wellbeing bio-monitoring.ca Scientific & Technical Innovation About us Mission & Vision Contact Search 0 Wishlist 0 Cart our product»Mission & Vision Create natural and sustainable products that enhance wellness — from laboratory breakthroughs to everyday use....

Result 3:
Mission: Our mission is to integrate biotechnology and nutrition for a healthier, more sustainable community — by producing natural, science-based alternatives to synthetic supplements and promotin